In [1]:
! uv pip install langchain-core langchain-community langgraph langchain_google_genai transformers pymongo openai langchain-core langchain-openai langgraph python-dotenv numpy sentence-transformers vllm

Using Python 3.12.12 environment at: /usr
Resolved 202 packages in 1.55s
⠙ Preparing packages... (0/68)
⠙ Preparing packages... (0/68)
⠙ Preparing packages... (0/68)
protobuf             ------------------------------     0 B/315.88 KiB
⠙ Preparing packages... (0/68)
protobuf             ------------------------------     0 B/315.88 KiB
⠙ Preparing packages... (0/68)
protobuf             ------------------------------     0 B/315.88 KiB
⠙ Preparing packages... (0/68)
lark                 ------------------------------     0 B/108.43 KiB
protobuf             ------------------------------     0 B/315.88 KiB
⠙ Preparing packages... (0/68)
lark                 ------------------------------     0 B/108.43 KiB
protobuf             ------------------------------     0 B/315.88 KiB
⠙ Preparing packages... (0/68)
lark                 ------------------------------     0 B/108.43 KiB
protobuf             ------------------------------     0 B/315.88 KiB
⠙ Preparing packages... (0/68)
lark     

In [2]:
import subprocess
import time
import socket
import requests
from openai import OpenAI

class Config:
    MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507"
    HOST = "0.0.0.0"
    PORT = 8000
    GPU_UTIL = 0.85
    MAX_LEN = 14000


def wait_for_server(host, port, timeout=600):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            with socket.create_connection((host, port), timeout=5):
                return True
        except OSError:
            time.sleep(5)
    return False


def start_vllm_background():
    cmd = [
        "vllm",
        "serve",
        Config.MODEL_NAME,
        "--host", Config.HOST,
        "--port", str(Config.PORT),
        "--gpu-memory-utilization", str(Config.GPU_UTIL),
        "--max-model-len", str(Config.MAX_LEN),
        "--dtype", "auto",          # H100 tối ưu bf16
        "--tensor-parallel-size", "1",  # single GPU H100
    ]

    print("Starting vLLM server...")
    
    # Mở file để ghi log thay vì dùng PIPE
    vllm_stdout = open("vllm_stdout.log", "w")
    vllm_stderr = open("vllm_stderr.log", "w")
    
    process = subprocess.Popen(
        cmd,
        stdout=vllm_stdout,   # Chuyển output ra file
        stderr=vllm_stderr,   # Chuyển error ra file
        text=True,
    )

    if not wait_for_server("localhost", Config.PORT):
        stderr = process.stderr.read()
        raise RuntimeError(f"vLLM failed to start:\n{stderr}")

    print("vLLM server is ready.")
    return process
start_vllm_background()

Starting vLLM server...
vLLM server is ready.


<Popen: returncode: None args: ['vllm', 'serve', 'Qwen/Qwen3-30B-A3B-Instruc...>

In [ ]:
from langgraph.graph import StateGraph, START, END
import os
from dotenv import load_dotenv
from typing import Dict, Optional
from langchain_core.messages import AIMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import json
load_dotenv(override=True)
from typing_extensions import TypedDict
from openai import OpenAI
import os

# Khởi tạo client 1 lần duy nhất
openai_client = OpenAI(
    base_url="http://localhost:8000/v1",  # vLLM server
    api_key="EMPTY"  # vLLM không cần key thật
)

def call_model(prompt: str,
               model: str = "Qwen/Qwen3-30B-A3B-Instruct-2507",
               temperature: float = 0.0,
               max_tokens: int = 4096) -> str:
    """
    Gọi vLLM server theo chuẩn OpenAI ChatCompletion.
    Trả về nội dung text thuần.
    """

    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content.strip()
class Message(TypedDict):
    subject: str
    question: str
    reasoning: str
    answer: str
    result: str = ""
    

def combine_convert(text: str, subject: str = "chemical") -> str:
    """Hàm tổng hợp gộp tất cả các bước xử lý: loại bỏ cụm từ thừa, chuẩn hóa endline, convert LaTeX, loại bỏ đáp án thừa."""
    # Tạo phần hướng dẫn cho bước 3 tùy theo môn học
    if subject == "chemical":
        step3_guide = """**ÁP DỤNG BƯỚC NÀY** - Convert các công thức hóa học sang LaTeX:

CẤU HÌNH ELECTRON với dấu \\n:
- SAI: '1s\\n2\\n2s\\n2\\n2p\\n6' → ĐÚNG: '\\\\(1s^2 2s^2 2p^6\\\\)' (BẮT BUỘC dùng LaTeX)
- SAI: '1s\\n2\\n2s\\n2\\n2p\\n3' → ĐÚNG: '\\\\(1s^2 2s^2 2p^3\\\\)'

CÔNG THỨC HÓA HỌC:
- SAI: 'CuCl\\n2', 'H\\n2\\nO', 'SO\\n2', 'CO\\n2', 'R\\n2\\nO\\n7'
- ĐÚNG: '\\\\(CuCl_2\\\\)', '\\\\(H_2O\\\\)', '\\\\(SO_2\\\\)', '\\\\(CO_2\\\\)', '\\\\(R_2O_7\\\\)' (BẮT BUỘC dùng LaTeX)

SỐ KHỐI: '\\n63\\nCu' → '\\\\(^{{63}}Cu\\\\)', '\\n24\\nMg' → '\\\\(^{{24}}Mg\\\\)'
KÝ HIỆU: 'M\\nCl' → '\\\\(M_{{Cl}}\\\\)', 'r\\nLi' → '\\\\(r_{{Li}}\\\\)', 'Z\\nNa' → '\\\\(Z_{{Na}}\\\\)'
ION: 'Fe\\n2+' → '\\\\(Fe^{{2+}}\\\\)', 'O\\n2-' → '\\\\(O^{{2-}}\\\\)', 'Na\\n+' → '\\\\(Na^+\\\\)'"""
    else:
        step3_guide = "**BỎ QUA BƯỚC NÀY** - Không phải môn hóa học"
    
    prompt = f"""Nhiệm vụ: Xử lý và chuẩn hóa text theo 4 BƯỚC SAU (thực hiện TẤT CẢ các bước):

Text cần xử lý:
\"\"\"
{text}
\"\"\"

===== BƯỚC 1: LOẠI BỎ CỤM TỪ THỪA =====
Loại bỏ các cụm từ thừa sau (nếu có):
- "II. Tự luận", "I. Tự luận"
- "I. PHẦN TRẮC NGHIỆM", "II. PHẦN TRẮC NGHIỆM", "III. PHẦN TRẮC NGHIỆM"
- "Phương pháp giải:", "Cách giải:", "Lời giải:", "Hướng dẫn giải:"
- "Trắc nghiệm Đúng/Sai", "Trắc nghiệm ngắn", "Trắc nghiệm nhiều đáp án"
- "Câu hỏi:", dấu "*" đầu dòng thừa, dấu '()' thừa
Ngoài ra có thể loại bỏ thêm các cụm từ thừa khác không liên quan đến nội dung chính.

===== BƯỚC 2: CHUẨN HÓA XUỐNG DÒNG VÀ KHOẢNG TRẮNG =====
- Loại bỏ nhiều dấu \\n liên tiếp (\\n\\n\\n → \\n)
- Loại bỏ dấu \\n thừa ở đầu/cuối text
- Loại bỏ nhiều khoảng trắng liên tiếp (    → 1 space)
- Loại bỏ khoảng trắng ở đầu/cuối mỗi dòng
- Loại bỏ khoảng trắng thừa trước dấu câu: "Câu 1 ." → "Câu 1."
- Giữ tối đa 1 dòng trống giữa các đoạn logic
- Loại bỏ dấu : thừa

===== BƯỚC 3: CONVERT CÔNG THỨC LATEX (CHỈ nếu subject="chemical") =====
{step3_guide}

# ===== BƯỚC 4: ĐẢM BẢO CẤU TRÚC [GIẢI THÍCH] → [ĐÁP ÁN] =====

## 🎯 Mục tiêu
Chuẩn hóa nội dung theo thứ tự bắt buộc:

**[Giải thích] → [Đáp án]**

---

## 🚨 NGOẠI LỆ DUY NHẤT – KHÔNG ĐƯỢC XỬ LÝ

KHÔNG chỉnh sửa nếu nội dung thuộc đúng một trong hai dạng sau:

- `Đáp án A. Sai vì ...`
- `Đáp án A.\nSai vì ...`

Tức là:
- Đáp án đứng ở đầu
- Ngay sau đó là phần giải thích trực tiếp cho chính đáp án đó
- Không có sự tách biệt giữa đáp án và phần giải thích

⚠️ Không được mở rộng thêm bất kỳ ngoại lệ nào ngoài hai trường hợp trên.

---

## 🔎 Quy tắc xử lý

### 1️⃣ Nếu cùng một đáp án xuất hiện ở cả ĐẦU và CUỐI nội dung:

- XÓA phần đáp án ở đầu
- GIỮ phần đáp án ở cuối
 
=================================================
QUY TẮC QUAN TRỌNG (ÁP DỤNG CHO TẤT CẢ CÁC BƯỚC):
1. GIỮ NGUYÊN 100% các công thức LaTeX đã có trong \\( ... \\)
2. GIỮ NGUYÊN các ký hiệu đặc biệt: °C, →, ⇔, ≥, ≤
3. GIỮ NGUYÊN tất cả nội dung chính, CHỈ làm sạch format
4. KHÔNG thêm, bớt, sửa hoặc giải thích gì
5. Thực hiện TUẦN TỰ từ bước 1 → 2 → 3 → 4
6. Nếu text là câu hỏi thì chỉ thực hiện 3 bước đầu, BƯỚC 4 CHỈ ÁP DỤNG CHO PHẦN LỜI GIẢI
7. Nếu text là câu hỏi thì không được thêm bất kỳ phần nào như [GIẢI THÍCH], [ĐÁP ÁN], hoặc các tiêu đề khác, chỉ làm sạch format theo 3 bước đầu và giữ nguyên nội dung chính

Output: Chỉ trả về text đã xử lý qua TẤT CẢ 4 bước, KHÔNG kèm giải thích."""

    combined_text = call_model(prompt)
    return combined_text.strip()

def critic(question: str, reasoning: str) -> str:
    """Đánh giá tính liền mạch giữa câu hỏi và lời giải."""
    prompt = f"""Nhiệm vụ: Đánh giá xem câu hỏi và lời giải có liền mạch, hợp lý không.

Câu hỏi:
\"\"\"
{question}
\"\"\"

Lời giải:
\"\"\"
{reasoning}
\"\"\"

TIÊU CHÍ ĐÁNH GIÁ:

1. KIỂM TRA DUPLICATE:
   - Nếu câu hỏi xuất hiện LẶP LẠI 2 LẦN (hoặc nhiều hơn) trong phần lời giải → Trả về: duplicate
   
2. KIỂM TRA TÍNH LIỀN MẠCH:
   - Lời giải có trả lời ĐÚNG câu hỏi không?
   - Lời giải có logic, mạch lạc không?
   - Nội dung có nhảy cóc, thiếu liên kết không?
   - Nếu KHÔNG liền mạch → Trả về: fail
   
3. Nếu qua được 2 bước trên → Trả về: success

QUY TẮC:
- CHỈ trả về MỘT trong ba giá trị: duplicate, fail, hoặc success
- KHÔNG giải thích, KHÔNG thêm bất kỳ text nào khác
- Ưu tiên kiểm tra duplicate trước, sau đó mới kiểm tra tính liền mạch

Output: Chỉ trả về MỘT từ: duplicate hoặc fail hoặc success"""

    result = call_model(prompt)
    return result.strip().lower()

def combine_convert_node(stage: Message) -> Message:
    """Node tổng hợp - gộp tất cả các bước xử lý trong 1 lần gọi LLM."""
    return Message(
        subject=stage["subject"],
        question=combine_convert(stage["question"], stage["subject"]),
        reasoning=combine_convert(stage["reasoning"], stage["subject"]) if  "Sai" not in stage["reasoning"] else stage["reasoning"],
        answer=stage["answer"],
        result=stage["result"]
    )

def critic_node(stage: Message) -> Message:
    """Node đánh giá tính liền mạch giữa câu hỏi và lời giải."""
    return Message(
        subject=stage["subject"],
        question=stage["question"],
        reasoning=stage["reasoning"],
        answer=stage["answer"],
        result=critic(stage["question"], stage["reasoning"])
    )
   
agent_graph = StateGraph(Message)
agent_graph.add_node("clean_combine", combine_convert_node)
agent_graph.add_node("critic", critic_node)
agent_graph.add_edge(START, "clean_combine")
agent_graph.add_edge("clean_combine", "critic")
agent_graph.add_edge("critic", END)
agent = agent_graph.compile()
with open("/kaggle/input/datasets/hoaletuyet/loigiaihay/loigiaihay_cleaned.json", "r", encoding="utf-8") as f:
    data = json.load(f)
with open("loigiaihay_final.json", "w", encoding="utf-8") as f:
    f.write("[\n")  # Bắt đầu mảng JSON
for i in range(len(data)):
    item = data[i]
    print(f"Processing item {i+1}/{len(data)}: {item['subject']}")
    message = Message(
        subject=item["subject"],
        question=item["question"],
        reasoning=item["reasoning"],
        answer=item["answer"],
        result=""
    )
    result = agent.invoke(message)
    with open("loigiaihay_final.json", "a", encoding="utf-8") as f:
        json.dump({
            "subject": result["subject"],
            "question": result["question"],
            "reasoning": result["reasoning"],
            "answer": result["answer"],
            "result": result["result"]
        }, f, ensure_ascii=False, indent=4)
        f.write(",\n" if i < len(data) - 1 else "\n")  # Thêm dấu phẩy giữa các phần tử, trừ phần tử cuối
with open("loigiaihay_final.json", "a", encoding="utf-8") as f:
    f.write("]")  # Kết thúc mảng JSON

Processing item 1/5334: chemical
Processing item 2/5334: chemical
Processing item 3/5334: chemical
Processing item 4/5334: chemical
Processing item 5/5334: chemical
Processing item 6/5334: chemical
Processing item 7/5334: chemical
Processing item 8/5334: chemical
Processing item 9/5334: chemical
Processing item 10/5334: chemical
Processing item 11/5334: chemical
Processing item 12/5334: chemical
Processing item 13/5334: chemical
Processing item 14/5334: chemical
Processing item 15/5334: chemical
Processing item 16/5334: chemical
Processing item 17/5334: chemical
Processing item 18/5334: chemical
Processing item 19/5334: chemical
Processing item 20/5334: chemical
Processing item 21/5334: chemical
Processing item 22/5334: chemical
Processing item 23/5334: chemical
Processing item 24/5334: chemical
Processing item 25/5334: chemical
Processing item 26/5334: chemical
Processing item 27/5334: chemical
Processing item 28/5334: chemical
Processing item 29/5334: chemical
Processing item 30/5334